In [ ]:
import pandas as pd

# Load main train data
df = pd.read_csv("train.csv")
print("=== SHAPE ===")
print(df.shape)

print("\n=== DTYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== SAMPLE ROWS ===")
print(df.head())

print("\n=== UNIQUE POS ===")
print(df['nama_pos'].nunique())
print(df['nama_pos'].unique())

print("\n=== DATETIME RANGE ===")
print(df['datetime'].min(), "-", df['datetime'].max())

In [ ]:
# Convert datetime buat analisis (temporary, exploration only)
df['datetime_parsed'] = pd.to_datetime(df['datetime'])

# 1. Cek jumlah baris per pos
print(" ROW COUNT PER POS ")
print(df.groupby('nama_pos').size().sort_values())

# 2. Cek range tanggal per pos (min-max)
print("\n DATE RANGE PER POS ")
range_per_pos = df.groupby('nama_pos')['datetime_parsed'].agg(['min', 'max'])
print(range_per_pos)

# 3. Load test.csv buat cek rentang tanggalnya
test = pd.read_csv("test.csv")
print("\n TEST SHAPE ")
print(test.shape)
print("\n TEST SAMPLE ")
print(test.head())

In [ ]:
import numpy as np

# Fokus ke 3 pos anomali
pos_anomali = ['Gunungsari', 'Floodway Bridge C', 'Bojonegoro - Kali Kethek']

for pos in pos_anomali:
    print(f"\n{''*60}")
    print(f"POS: {pos}")
    print(''*60)
    
    sub = df[df['nama_pos'] == pos].sort_values('datetime_parsed')
    
    # Buat expected timestamp grid: dari min ke max, jam 06/12/18 tiap hari
    start = sub['datetime_parsed'].min().normalize()
    end = sub['datetime_parsed'].max().normalize()
    all_dates = pd.date_range(start, end, freq='D')
    
    expected_ts = []
    for d in all_dates:
        for h in [6, 12, 18]:
            expected_ts.append(d + pd.Timedelta(hours=h))
    expected_ts = pd.Series(expected_ts)
    
    actual_ts = set(sub['datetime_parsed'])
    missing_ts = expected_ts[~expected_ts.isin(actual_ts)]
    
    print(f"Expected observations: {len(expected_ts)}")
    print(f"Actual observations: {len(sub)}")
    print(f"Missing: {len(missing_ts)}")
    
    if len(missing_ts) > 0:
        # Cek distribusi missing per bulan
        missing_df = pd.DataFrame({'ts': missing_ts})
        missing_df['year_month'] = missing_df['ts'].dt.to_period('M')
        print("\nMissing count per bulan:")
        print(missing_df['year_month'].value_counts().sort_index())
        
        # Cek apakah gap-nya berturutan (consecutive) atau tersebar
        missing_dates_only = missing_df['ts'].dt.date.unique()
        missing_dates_only = sorted(missing_dates_only)
        print(f"\nJumlah hari unik yang punya minimal 1 missing: {len(missing_dates_only)}")
        print(f"Tanggal missing pertama: {missing_dates_only[0]}")
        print(f"Tanggal missing terakhir: {missing_dates_only[-1]}")

In [ ]:
# Cek apakah gap Februari 2025 ini terjadi di SEMUA 30 pos, bukan cuma 3 ini
feb2025 = df[(df['datetime_parsed'] >= '2025-02-01') & (df['datetime_parsed'] < '2025-03-01')]

print(" JUMLAH OBSERVASI PER POS DI FEBRUARI 2025 ")
feb_counts = feb2025.groupby('nama_pos').size().sort_values()
print(feb_counts)

expected_feb = 28 * 3  # 28 hari x 3 observasi
print(f"\nEkspektasi normal per pos: {expected_feb}")
print(f"\nPos dengan observasi Feb 2025 jauh di bawah ekspektasi:")
print(feb_counts[feb_counts < expected_feb * 0.8])

In [ ]:
# Cek tanggal persis mana yang masih ada data di Feb 2025
print(feb2025.groupby(feb2025['datetime_parsed'].dt.date).size())

In [ ]:
print("=== OVERALL TARGET STATS ===")
print(df['tma_mdpl'].describe())

print("\n=== SKEWNESS & KURTOSIS (overall) ===")
print("Skewness:", df['tma_mdpl'].skew())
print("Kurtosis:", df['tma_mdpl'].kurtosis())

print("\n=== STATS PER POS ===")
stats_per_pos = df.groupby('nama_pos')['tma_mdpl'].agg(
    ['count', 'mean', 'std', 'min', 'max', 'skew']
).sort_values('mean', ascending=False)
print(stats_per_pos)

print("\n=== EXTREME VALUES CHECK ===")
# Cek nilai <= 0 (fisik gak masuk akal buat TMA) dan outlier ekstrem
print("Jumlah tma_mdpl <= 0:", (df['tma_mdpl'] <= 0).sum())
print(df[df['tma_mdpl'] <= 0][['datetime', 'nama_pos', 'tma_mdpl']])

print("\n=== TOP 10 NILAI TERTINGGI ===")
print(df.nlargest(10, 'tma_mdpl')[['datetime', 'nama_pos', 'tma_mdpl']])

In [ ]:
import matplotlib.pyplot as plt

# Ambil ulang skewness per pos dari data asli (bukan hardcode)
skew_per_pos = df.groupby('nama_pos')['tma_mdpl'].skew().sort_values()

# Warna: biru buat negative-skew (anjlok-prone), merah buat positive-skew (spike-prone)
colors = ['#2a78d6' if v < 0 else '#e34948' for v in skew_per_pos.values]

fig, ax = plt.subplots(figsize=(9, 10))
bars = ax.barh(skew_per_pos.index, skew_per_pos.values, color=colors)

ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Skewness tma_mdpl')
ax.set_title('Skewness distribusi TMA per pos pemantauan')
ax.grid(axis='x', linestyle='--', alpha=0.4)

# Tambah label angka di ujung tiap bar biar gampang dibaca
for bar, val in zip(bars, skew_per_pos.values):
    ax.text(val + (1 if val >= 0 else -1), bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
env = pd.read_csv("data_pendukung/data_lingkungan.csv")

print("=== SHAPE ===")
print(env.shape)

print("\n=== DTYPES ===")
print(env.dtypes)

print("\n=== MISSING VALUES ===")
print(env.isnull().sum())

print("\n=== SAMPLE ROWS ===")
print(env.head())

# Cek granularity datetime — beneran per jam?
env['datetime_parsed'] = pd.to_datetime(env['datetime'])
print("\n=== DATETIME RANGE ===")
print(env['datetime_parsed'].min(), "-", env['datetime_parsed'].max())

print("\n=== JUMLAH POS DI ENV ===")
print(env['nama_pos'].nunique())

# Cek ID uniqueness — kombinasi datetime + nama_pos harusnya unik
print("\n=== DUPLIKAT (datetime + nama_pos) ===")
print(env.duplicated(subset=['datetime', 'nama_pos']).sum())

print("\n=== CARDINALITY KOLOM NON-NUMERIK ===")
cat_cols = env.select_dtypes(include=['object']).columns
for col in cat_cols:
    print(f"{col}: {env[col].nunique()} unique values")

In [ ]:
# 1. Cek persis tanggal mana yang kena missing 720-baris (soil moisture, pressure, MJO)
missing_soil = env[env['soil_moisture_0_7cm'].isnull()]
print("=== TANGGAL MISSING SOIL/PRESSURE/MJO ===")
print(missing_soil['datetime_parsed'].dt.date.value_counts())

# 2. Cek nino_34 missing — tanggal mana aja
missing_nino = env[env['nino_34'].isnull()]
print("\n=== RENTANG TANGGAL MISSING NINO_34 ===")
print(missing_nino['datetime_parsed'].min(), "-", missing_nino['datetime_parsed'].max())
print("Jumlah bulan unik yang missing:", missing_nino['datetime_parsed'].dt.to_period('M').nunique())

# 3. Cek solar_radiation — apakah beneran ada sentinel -999
print("\n=== SOLAR RADIATION VALUE CHECK ===")
print(env['solar_radiation_mj_m2'].describe())
print("\nJumlah nilai -999:", (env['solar_radiation_mj_m2'] == -999).sum())
print("Jumlah nilai < 0 (selain -999):", ((env['solar_radiation_mj_m2'] < 0) & (env['solar_radiation_mj_m2'] != -999)).sum())

# 4. Cek rainfall_mm vs rainfall_openmeteo_mm — beneran duplikat 100%?
print("\n=== RAINFALL DUPLICATE CHECK ===")
print("Jumlah baris beda:", (env['rainfall_mm'] != env['rainfall_openmeteo_mm']).sum())

In [ ]:
missing_solar = env[env['solar_radiation_mj_m2'] == -999]

print("=== RENTANG TANGGAL SOLAR -999 ===")
print(missing_solar['datetime_parsed'].min(), "-", missing_solar['datetime_parsed'].max())

print("\n=== JUMLAH -999 PER BULAN ===")
print(missing_solar['datetime_parsed'].dt.to_period('M').value_counts().sort_index())

# Cek juga apakah -999 ini terjadi di semua pos atau cuma sebagian
print("\n=== JUMLAH -999 PER POS (cek 5 teratas & terbawah) ===")
per_pos_missing = missing_solar.groupby('nama_pos').size().sort_values()
print(per_pos_missing)

In [ ]:
numeric_cols = [
    'rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c',
    'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_max_24h_mm',
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm',
    'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa'
]

print("=== DESCRIBE FITUR NUMERIK ===")
print(env[numeric_cols].describe().T)

print("\n=== SKEWNESS ===")
print(env[numeric_cols].skew().sort_values())

# Cek khusus rainfall — biasanya sangat skewed (banyak nol, sesekali ekstrem)
print("\n=== RAINFALL_MM: PROPORSI NOL ===")
print((env['rainfall_mm'] == 0).mean())

print("\n=== RAINFALL_MM: TOP 10 EKSTREM ===")
print(env.nlargest(10, 'rainfall_mm')[['datetime', 'nama_pos', 'rainfall_mm']])

# Cek soil moisture — apakah rentangnya masuk akal (biasanya 0-1 m3/m3)
print("\n=== SOIL MOISTURE RANGE CHECK ===")
for col in ['soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']:
    print(f"{col}: min={env[col].min():.4f}, max={env[col].max():.4f}")

# Cek surface_pressure vs pressure_msl — beda jauh gak? (indikasi elevasi seperti dicurigai sebelumnya)
print("\n=== PRESSURE COMPARISON PER POS (mean) ===")
pressure_check = env.groupby('nama_pos')[['surface_pressure_hpa', 'pressure_msl_hpa']].mean()
pressure_check['diff'] = pressure_check['pressure_msl_hpa'] - pressure_check['surface_pressure_hpa']
print(pressure_check.sort_values('diff'))

In [ ]:
# 1. Wind direction — ini circular data (0° = 360°), histogram biasa bisa menyesatkan
print("=== WIND DIRECTION DISTRIBUTION ===")
print(env['wind_direction_deg'].describe())

# Cek apakah ada pola dominan (arah angin tertentu lebih sering)
import numpy as np
bins = [0, 45, 90, 135, 180, 225, 270, 315, 360]
labels = ['N-NE', 'NE-E', 'E-SE', 'SE-S', 'S-SW', 'SW-W', 'W-NW', 'NW-N']
env['wind_dir_bin'] = pd.cut(env['wind_direction_deg'], bins=bins, labels=labels, include_lowest=True)
print("\n=== DISTRIBUSI ARAH ANGIN (BINNED) ===")
print(env['wind_dir_bin'].value_counts())

# 2. MJO — cek granularity asli, apakah beneran berubah per hari (bukan per jam)
print("\n=== MJO — CEK GRANULARITY (1 pos, 3 hari pertama) ===")
sample_mjo = env[env['nama_pos'] == 'Arjowinangun - Pacitan'].head(72)[['datetime', 'mjo_phase', 'mjo_amplitude', 'rmm1', 'rmm2']]
print(sample_mjo.drop_duplicates(subset=['mjo_phase', 'mjo_amplitude', 'rmm1', 'rmm2']))

# 3. Cek jumlah nilai unik MJO per hari — harusnya 1 kalau resolusi asli harian
env['date_only'] = env['datetime_parsed'].dt.date
mjo_check = env.groupby(['nama_pos', 'date_only'])['mjo_phase'].nunique()
print("\n=== JUMLAH NILAI MJO_PHASE UNIK PER (POS, TANGGAL) ===")
print(mjo_check.value_counts())

# 4. Cek nino_34 — apakah beneran konstan per bulan
nino_check = env.groupby(['nama_pos', env['datetime_parsed'].dt.to_period('M')])['nino_34'].nunique()
print("\n=== JUMLAH NILAI NINO_34 UNIK PER (POS, BULAN) ===")
print(nino_check.value_counts())

# 5. Cek apakah MJO/nino_34 sama di semua pos pada waktu yang sama (harusnya ya, karena ini indeks global)
print("\n=== MJO_PHASE SAMA GAK ANTAR POS DI WAKTU YANG SAMA? ===")
sample_time = env[env['datetime'] == '2023-06-15 12:00:00'][['nama_pos', 'mjo_phase', 'nino_34']]
print(sample_time)

In [ ]:
# Filter env hanya di jam yang match target (06, 12, 18) — exact match, bukan agregasi
env_matched = env[env['datetime_parsed'].dt.hour.isin([6, 12, 18])].copy()

# Merge dengan train berdasarkan datetime + nama_pos
merged = df.merge(
    env_matched,
    on=['datetime', 'nama_pos'],
    how='inner',
    suffixes=('', '_env')
)

print("=== SHAPE SETELAH MERGE ===")
print(merged.shape)
print("Shape df asli:", df.shape)
print("Baris yang gak ke-match:", df.shape[0] - merged.shape[0])

# Cek korelasi GLOBAL dulu (buat dibandingkan sama within-pos nanti — expect ada spurious correlation)
numeric_features = [
    'rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c',
    'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_max_24h_mm',
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm',
    'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa',
    'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34'
]

print("\n=== KORELASI GLOBAL vs tma_mdpl ===")
global_corr = merged[numeric_features + ['tma_mdpl']].corr()['tma_mdpl'].drop('tma_mdpl').sort_values()
print(global_corr)

# Sekarang WITHIN-POS correlation — hitung korelasi per pos, lalu rata-rata
print("\n=== WITHIN-POS CORRELATION (rata-rata across 30 pos) ===")
within_pos_corrs = {}
for feat in numeric_features:
    corrs_per_pos = merged.groupby('nama_pos').apply(
        lambda g: g[feat].corr(g['tma_mdpl']) if g[feat].notna().sum() > 10 else np.nan
    )
    within_pos_corrs[feat] = corrs_per_pos.mean()

within_pos_series = pd.Series(within_pos_corrs).sort_values()
print(within_pos_series)

# Bandingkan global vs within-pos berdampingan
print("\n=== PERBANDINGAN GLOBAL vs WITHIN-POS ===")
comparison = pd.DataFrame({'global_corr': global_corr, 'within_pos_corr': within_pos_series})
print(comparison.sort_values('within_pos_corr', key=abs, ascending=False))

In [ ]:
# ALTERNATIF 1: Agregasi env ke rata-rata HARIAN, lalu match ke target per hari
# (bukan exact-hour match)
env_daily = env.groupby(['nama_pos', env['datetime_parsed'].dt.date])[numeric_features].mean().reset_index()
env_daily.columns = ['nama_pos', 'date_only'] + numeric_features

df_temp = df.copy()
df_temp['date_only'] = df_temp['datetime_parsed'].dt.date
merged_daily = df_temp.merge(env_daily, on=['nama_pos', 'date_only'], how='inner')

print("=== ALTERNATIF 1: Daily-mean aggregation ===")
within_pos_daily = {}
for feat in numeric_features:
    corrs = merged_daily.groupby('nama_pos').apply(
        lambda g: g[feat].corr(g['tma_mdpl']) if g[feat].notna().sum() > 10 else np.nan
    )
    within_pos_daily[feat] = corrs.mean()

daily_series = pd.Series(within_pos_daily).sort_values()
print(daily_series[['soil_moisture_7_28cm', 'surface_pressure_hpa']])

# ALTERNATIF 2: Pakai MEDIAN across-pos, bukan mean (exact-hour match, dari merged sebelumnya)
print("\n=== ALTERNATIF 2: Median across-pos (exact-hour match) ===")
within_pos_median = {}
for feat in numeric_features:
    corrs = merged.groupby('nama_pos').apply(
        lambda g: g[feat].corr(g['tma_mdpl']) if g[feat].notna().sum() > 10 else np.nan
    )
    within_pos_median[feat] = corrs.median()

median_series = pd.Series(within_pos_median).sort_values()
print(median_series[['soil_moisture_7_28cm', 'surface_pressure_hpa']])

# ALTERNATIF 3: Cek sebaran korelasi per pos individual — siapa tau ada 1-2 pos ekstrem yang narik mean
print("\n=== SEBARAN KORELASI PER POS: soil_moisture_7_28cm ===")
corrs_soil = merged.groupby('nama_pos').apply(
    lambda g: g['soil_moisture_7_28cm'].corr(g['tma_mdpl'])
).sort_values()
print(corrs_soil)

print("\n=== SEBARAN KORELASI PER POS: surface_pressure_hpa ===")
corrs_pressure = merged.groupby('nama_pos').apply(
    lambda g: g['surface_pressure_hpa'].corr(g['tma_mdpl'])
).sort_values()
print(corrs_pressure)

In [ ]:
# 1. Korelasi antar 4 layer soil moisture — cek redundansi
soil_cols = ['soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']
print("=== KORELASI ANTAR LAYER SOIL MOISTURE ===")
print(merged[soil_cols].corr())

# 2. Korelasi antar fitur atmosfer yang mungkin redundan (dew_point, humidity, cloud_cover sering overlap)
atmos_cols = ['dew_point_c', 'humidity_pct', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh']
print("\n=== KORELASI ANTAR FITUR ATMOSFER ===")
print(merged[atmos_cols].corr())

# 3. Full correlation matrix semua fitur numerik (buat scan menyeluruh pasangan tinggi)
full_corr = merged[numeric_features].corr()

# Ambil pasangan dengan |corr| tinggi (>0.7), exclude diagonal
print("\n=== PASANGAN FITUR DENGAN |KORELASI| > 0.7 (kandidat redundan) ===")
corr_pairs = full_corr.where(np.triu(np.ones(full_corr.shape), k=1).astype(bool)).stack()
high_corr = corr_pairs[abs(corr_pairs) > 0.7].sort_values(key=abs, ascending=False)
print(high_corr)

# 4. Surprising relationship check: rainfall_max_24h_mm vs soil moisture (harusnya positif kuat - hujan mengisi tanah)
print("\n=== RAINFALL_MAX_24H vs SOIL MOISTURE (within-pos, harusnya positif) ===")
rain_soil_corr = {}
for feat in soil_cols:
    corrs = merged.groupby('nama_pos').apply(
        lambda g: g['rainfall_max_24h_mm'].corr(g[feat])
    )
    rain_soil_corr[feat] = corrs.mean()
print(pd.Series(rain_soil_corr))

# 5. Cek wind_direction vs tma_mdpl per pos — apakah arah angin tertentu konsisten across pos (indikasi topografi/geografis)
print("\n=== WIND_DIRECTION_DEG korelasi per pos (cek konsistensi arah) ===")
wind_corr = merged.groupby('nama_pos').apply(
    lambda g: g['wind_direction_deg'].corr(g['tma_mdpl'])
).sort_values()
print(wind_corr)

In [ ]:
# 1. Autocorrelation lag-1, lag-2, lag-3 per pos — cek ulang dengan lebih detail
from pandas.plotting import autocorrelation_plot

print("=== AC LAG 1,2,3 PER POS (top 5 & bottom 5) ===")
ac_results = {}
for pos in df['nama_pos'].unique():
    sub = df[df['nama_pos'] == pos].sort_values('datetime_parsed')
    ac_results[pos] = {
        'lag1': sub['tma_mdpl'].autocorr(lag=1),
        'lag2': sub['tma_mdpl'].autocorr(lag=2),
        'lag3': sub['tma_mdpl'].autocorr(lag=3),
    }
ac_df = pd.DataFrame(ac_results).T.sort_values('lag1')
print(ac_df)

# 2. Cek APAKAH lag-1 di sekitar gap Feb 2025 itu "palsu" (lompat jauh secara waktu tapi dihitung sbg lag-1)
# Ambil 1 pos contoh, lihat time delta antar baris berurutan
sample_pos = df[df['nama_pos'] == 'Ngadipiro'].sort_values('datetime_parsed').copy()
sample_pos['time_delta_hours'] = sample_pos['datetime_parsed'].diff().dt.total_seconds() / 3600
print("\n=== TIME DELTA ANTAR BARIS BERURUTAN (Ngadipiro) ===")
print(sample_pos['time_delta_hours'].value_counts().sort_index())

print("\n=== BARIS DENGAN GAP TERBESAR (indikasi lompat lewat outage) ===")
print(sample_pos.nlargest(5, 'time_delta_hours')[['datetime', 'tma_mdpl', 'time_delta_hours']])

# 3. Cek pola musiman — rata-rata TMA per bulan (across semua pos, normalisasi per pos dulu)
df['month'] = df['datetime_parsed'].dt.month
df['year'] = df['datetime_parsed'].dt.year

# Normalisasi: TMA per pos dikurangi mean pos itu (biar bisa dibandingkan lintas pos meski beda skala)
df['tma_normalized'] = df.groupby('nama_pos')['tma_mdpl'].transform(lambda x: x - x.mean())

print("\n=== POLA MUSIMAN (TMA ternormalisasi, rata-rata per bulan) ===")
print(df.groupby('month')['tma_normalized'].mean())

# 4. Cek tren tahunan
print("\n=== RATA-RATA TMA TERNORMALISASI PER TAHUN ===")
print(df.groupby('year')['tma_normalized'].mean())

In [ ]:
# Filter hanya Jan-Sep di setiap tahun (periode yang ada di ketiga tahun)
df_fair = df[df['month'] <= 9].copy()

print("=== RATA-RATA TMA TERNORMALISASI PER TAHUN (Jan-Sep only, fair comparison) ===")
print(df_fair.groupby('year')['tma_normalized'].mean())

print("\n=== JUMLAH OBSERVASI PER TAHUN (Jan-Sep, cek balance) ===")
print(df_fair.groupby('year').size())

# Breakdown lebih detail: bulan x tahun, biar kelihatan pola per-bulan across tahun
print("\n=== TMA TERNORMALISASI: BULAN x TAHUN (Jan-Sep) ===")
pivot_check = df_fair.pivot_table(values='tma_normalized', index='month', columns='year', aggfunc='mean')
print(pivot_check)

# Cek juga per pos - siapa tau tren tahunan itu didorong oleh beberapa pos aja, bukan merata
print("\n=== TREN TAHUNAN PER POS (Jan-Sep, top 5 kenaikan & top 5 penurunan) ===")
pos_year_trend = df_fair.groupby(['nama_pos', 'year'])['tma_mdpl'].mean().unstack()
pos_year_trend['change_2023_to_2025'] = pos_year_trend[2025] - pos_year_trend[2023]
print(pos_year_trend.sort_values('change_2023_to_2025'))

In [ ]:
# FIX: split dari KIRI (n=1, split bukan rsplit), bukan dari kanan
test[['datetime_str', 'nama_pos_from_id']] = test['id'].str.split(' - ', n=1, expand=True)

print("=== VALIDASI ULANG SETELAH FIX ===")
print("Jumlah pos unik di test:", test['nama_pos_from_id'].nunique())
print("Pos di test tapi tidak ada di train:", 
      set(test['nama_pos_from_id'].unique()) - set(df['nama_pos'].unique()))
print("Pos di train tapi tidak ada di test:", 
      set(df['nama_pos'].unique()) - set(test['nama_pos_from_id'].unique()))

# Parse datetime ulang
test['datetime_parsed'] = pd.to_datetime(test['datetime_str'], format='%Y-%m-%d %H:%M:%S')
print("\n=== DATETIME RANGE TEST (setelah fix) ===")
print(test['datetime_parsed'].min(), "-", test['datetime_parsed'].max())

print("\n=== JUMLAH OBSERVASI TEST PER POS ===")
print(test.groupby('nama_pos_from_id').size().describe())

# Lanjut cek sisanya yang belum sempat jalan
print("\n=== IMPOSSIBLE VALUES CHECK (env) ===")
print("Rainfall negatif:", (env['rainfall_mm'] < 0).sum())
print("Humidity > 100 atau < 0:", ((env['humidity_pct'] > 100) | (env['humidity_pct'] < 0)).sum())
print("Cloud cover > 100 atau < 0:", ((env['cloud_cover_pct'] > 100) | (env['cloud_cover_pct'] < 0)).sum())
print("Wind direction > 360 atau < 0:", ((env['wind_direction_deg'] > 360) | (env['wind_direction_deg'] < 0)).sum())

sample_sub = pd.read_csv("sample_submission.csv")
print("\n=== SAMPLE SUBMISSION FORMAT ===")
print(sample_sub.shape)
print(sample_sub.head())
print(sample_sub.dtypes)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# =============================================================
# CELL EDA TAMBAHAN 1-6
# Asumsi: df, merged, env sudah ada dari EDA sebelumnya
# df       = train.csv + datetime_parsed + month + year + tma_normalized
# merged   = df merged dengan env di jam 06/12/18 (exact-hour)
# env      = data_lingkungan.csv + datetime_parsed + date_only
# =============================================================

print("=" * 70)
print("EDA TAMBAHAN 1: CROSS-CORRELATION LAG RAINFALL → TMA")
print("=" * 70)

lag_features = ['rainfall_mm', 'rainfall_max_24h_mm',
                'soil_moisture_7_28cm', 'soil_moisture_0_7cm']
max_lag = 10  # unit = 6 jam (lag-10 = 60 jam ke belakang)

lag_results = []
for pos in merged['nama_pos'].unique():
    sub = merged[merged['nama_pos'] == pos].sort_values('datetime_parsed').copy()
    for feat in lag_features:
        for lag in range(0, max_lag + 1):
            corr = sub[feat].shift(lag).corr(sub['tma_mdpl'])
            lag_results.append({
                'nama_pos': pos,
                'feature': feat,
                'lag_6h': lag,
                'lag_hours': lag * 6,
                'corr': corr
            })

lag_df = pd.DataFrame(lag_results)

# Rata-rata korelasi across semua pos per lag
print("\n--- Rata-rata |corr| across 30 pos per lag (tiap fitur) ---")
lag_mean = lag_df.groupby(['feature', 'lag_6h'])['corr'].mean().unstack(level=0)
print(lag_mean.round(3))

# Lag optimal per fitur (lag dengan corr rata-rata tertinggi)
print("\n--- Lag optimal per fitur (corr rata-rata tertinggi) ---")
for feat in lag_features:
    sub_feat = lag_df[lag_df['feature'] == feat]
    best = sub_feat.groupby('lag_6h')['corr'].mean().idxmax()
    best_corr = sub_feat.groupby('lag_6h')['corr'].mean().max()
    print(f"{feat:35s} → lag {best:2d} ({best * 6:3d} jam) | corr = {best_corr:.4f}")

# Per pos: lag optimal rainfall_max_24h_mm
print("\n--- Lag optimal rainfall_max_24h_mm per pos ---")
rain_lag = lag_df[lag_df['feature'] == 'rainfall_max_24h_mm']
best_lag_per_pos = rain_lag.loc[rain_lag.groupby('nama_pos')['corr'].idxmax()][
    ['nama_pos', 'lag_6h', 'lag_hours', 'corr']
].sort_values('lag_hours')
print(best_lag_per_pos.to_string(index=False))


print("\n" + "=" * 70)
print("EDA TAMBAHAN 2: KORELASI SPASIAL ANTAR POS (HULU-HILIR)")
print("=" * 70)

# Pivot TMA ke wide format (baris = datetime, kolom = pos)
tma_wide = df.pivot_table(index='datetime_parsed', columns='nama_pos', values='tma_mdpl')

print("\n--- Korelasi TMA antar pos (lag-0, contemporaneous) ---")
tma_corr = tma_wide.corr()
print(tma_corr.round(3))

# Pasangan dengan korelasi tertinggi (kandidat hulu-hilir)
print("\n--- Top 20 pasangan pos dengan korelasi TMA tertinggi ---")
corr_stack = tma_corr.where(np.triu(np.ones(tma_corr.shape), k=1).astype(bool)).stack()
corr_stack.index.names = ['pos_a', 'pos_b']  # rename dulu sebelum reset_index
corr_pairs = corr_stack.reset_index()
corr_pairs.columns = ['pos_a', 'pos_b', 'corr']
corr_pairs = corr_pairs.reindex(corr_pairs['corr'].abs().sort_values(ascending=False).index)
print(corr_pairs.head(20).to_string(index=False))

# Cross-lag: cek apakah TMA pos A lag-N jam bisa prediksi pos B
# (indikasi aliran hulu-hilir)
print("\n--- Cross-lag TMA: top 10 pasangan corr tertinggi, dicek per lag ---")
top_pairs = corr_pairs.head(10)[['pos_a', 'pos_b']].values

cross_lag_results = []
for pos_a, pos_b in top_pairs:
    ts_a = tma_wide[pos_a].dropna()
    ts_b = tma_wide[pos_b].dropna()
    common_idx = ts_a.index.intersection(ts_b.index)
    ts_a = ts_a.loc[common_idx]
    ts_b = ts_b.loc[common_idx]
    for lag in range(0, 9):  # lag 0-8 step (0-48 jam)
        corr = ts_a.shift(lag).corr(ts_b)
        cross_lag_results.append({
            'pos_a': pos_a, 'pos_b': pos_b,
            'lag_6h': lag, 'lag_hours': lag * 6, 'corr': corr
        })

cross_lag_df = pd.DataFrame(cross_lag_results)
best_cross = cross_lag_df.loc[cross_lag_df.groupby(['pos_a', 'pos_b'])['corr'].idxmax()]
print(best_cross[['pos_a', 'pos_b', 'lag_6h', 'lag_hours', 'corr']].to_string(index=False))


print("\n" + "=" * 70)
print("EDA TAMBAHAN 3: COVERAGE BULAN TRAIN VS TEST (EXTRAPOLATION RISK)")
print("=" * 70)

df['month'] = df['datetime_parsed'].dt.month
df['year']  = df['datetime_parsed'].dt.year

# Bulan yang harus diprediksi di test (Sep 2025 - Mei 2026)
test_months = list(range(9, 13)) + list(range(1, 6))  # 9,10,11,12,1,2,3,4,5

print("\n--- Jumlah observasi per (bulan, tahun) di train ---")
month_year_counts = df.groupby(['year', 'month']).size().unstack(fill_value=0)
print(month_year_counts)

print("\n--- Coverage bulan di train: berapa tahun punya data bulan tsb ---")
month_coverage = df.groupby('month')['year'].nunique().rename('n_years_in_train')
month_obs      = df.groupby('month').size().rename('total_obs')
month_summary  = pd.concat([month_coverage, month_obs], axis=1)
month_summary['in_test_period'] = month_summary.index.isin(test_months)
print(month_summary.sort_index())

print("\n--- Bulan di test period: risiko extrapolation ---")
for m in sorted(test_months):
    n_years = month_coverage.get(m, 0)
    n_obs   = month_obs.get(m, 0)
    risk    = "🟢 OK" if n_years >= 2 else ("🟡 TIPIS" if n_years == 1 else "🔴 ZERO")
    print(f"Bulan {m:2d} → {n_years} tahun di train ({n_obs:5d} obs) {risk}")

print("\n--- Per pos: cek bulan yang SAMA SEKALI tidak ada di train ---")
pos_month_coverage = df.groupby(['nama_pos', 'month']).size().unstack(fill_value=0)
zero_months = {}
for pos in pos_month_coverage.index:
    missing = [m for m in test_months if pos_month_coverage.loc[pos, m] == 0]
    if missing:
        zero_months[pos] = missing
if zero_months:
    for pos, months in zero_months.items():
        print(f"  ⚠️  {pos}: bulan {months} tidak ada di train")
else:
    print("  ✅ Semua pos punya setidaknya 1 observasi di setiap bulan test period")


print("\n" + "=" * 70)
print("EDA TAMBAHAN 4: OUTLIER — SPIKE SESAAT vs SUSTAINED (BANJIR NYATA)")
print("=" * 70)

spike_pos = [
    'Kali Anyar - Kreteg Abang', 'Napel', 'Kedungupit',
    'Jarum', 'Peren', 'Karangnongko', 'Bojonegoro - Kali Kethek'
]

def run_length_encoding(series):
    """Hitung panjang consecutive run dari True values."""
    runs = []
    count = 0
    for val in series:
        if val:
            count += 1
        else:
            if count > 0:
                runs.append(count)
            count = 0
    if count > 0:
        runs.append(count)
    return runs

print("\n--- Analisis spike per pos (q99 threshold) ---")
spike_summary = []
for pos in spike_pos:
    sub = df[df['nama_pos'] == pos].sort_values('datetime_parsed').copy()
    q95  = sub['tma_mdpl'].quantile(0.95)
    q99  = sub['tma_mdpl'].quantile(0.99)
    q999 = sub['tma_mdpl'].quantile(0.999)

    sub['is_extreme'] = sub['tma_mdpl'] > q99
    runs = run_length_encoding(sub['is_extreme'].values)

    n_extreme     = sub['is_extreme'].sum()
    n_runs        = len(runs)
    max_run       = max(runs) if runs else 0
    mean_run      = np.mean(runs) if runs else 0
    isolated      = sum(1 for r in runs if r == 1)

    print(f"\n{pos}")
    print(f"  q95={q95:.2f}, q99={q99:.2f}, q999={q999:.2f}")
    print(f"  Total extreme obs (>q99): {n_extreme}")
    print(f"  Jumlah run (episode):     {n_runs}")
    print(f"  Run terpanjang:           {max_run} obs ({max_run * 6} jam)")
    print(f"  Rata-rata run:            {mean_run:.1f} obs")
    print(f"  Spike isolated (run=1):   {isolated} ({isolated/n_runs*100:.0f}% dari episode)" if n_runs > 0 else "")

    spike_summary.append({
        'nama_pos': pos, 'q99': q99, 'n_extreme': n_extreme,
        'n_episodes': n_runs, 'max_run_obs': max_run,
        'max_run_hours': max_run * 6, 'isolated_pct': isolated / n_runs * 100 if n_runs else 0
    })

print("\n--- Ringkasan spike summary ---")
print(pd.DataFrame(spike_summary).set_index('nama_pos').round(1).to_string())


print("\n" + "=" * 70)
print("EDA TAMBAHAN 5: PERILAKU TMA DI SEKITAR GAP FEB 2025")
print("=" * 70)

gap_start = pd.Timestamp('2025-02-04')
gap_end   = pd.Timestamp('2025-03-01')
n_before  = 5
n_after   = 5

print(f"\nGap window: {gap_start.date()} s.d. {gap_end.date()}")
print(f"Melihat {n_before} baris sebelum gap & {n_after} baris setelah gap\n")

jump_results = []
for pos in df['nama_pos'].unique():
    sub = df[df['nama_pos'] == pos].sort_values('datetime_parsed').copy()

    before = sub[sub['datetime_parsed'] < gap_start].tail(n_before)
    after  = sub[sub['datetime_parsed'] >= gap_end].head(n_after)

    if len(before) == 0 or len(after) == 0:
        continue

    last_before = before['tma_mdpl'].iloc[-1]
    first_after = after['tma_mdpl'].iloc[0]
    jump        = first_after - last_before
    pct_change  = jump / last_before * 100 if last_before != 0 else np.nan

    jump_results.append({
        'nama_pos':    pos,
        'last_before': last_before,
        'first_after': first_after,
        'jump':        jump,
        'pct_change':  pct_change
    })

    # Print detail untuk pos dengan lompatan besar (>10% atau >2 mdpl)
    if abs(jump) > 2 or abs(pct_change) > 10:
        print(f"⚠️  {pos} | jump = {jump:+.3f} mdpl ({pct_change:+.1f}%)")
        print("   Before gap:")
        print(before[['datetime', 'tma_mdpl']].to_string(index=False))
        print("   After gap:")
        print(after[['datetime', 'tma_mdpl']].to_string(index=False))
        print()

jump_df = pd.DataFrame(jump_results).sort_values('jump', key=abs, ascending=False)

print("\n--- Ringkasan lompatan TMA di semua pos (sorted by |jump|) ---")
print(jump_df.round(3).to_string(index=False))

print(f"\n--- Summary statistik lompatan ---")
print(f"Rata-rata |jump|: {jump_df['jump'].abs().mean():.3f} mdpl")
print(f"Max |jump|:       {jump_df['jump'].abs().max():.3f} mdpl ({jump_df.loc[jump_df['jump'].abs().idxmax(), 'nama_pos']})")
print(f"Pos dengan |jump| > 2 mdpl:  {(jump_df['jump'].abs() > 2).sum()}")
print(f"Pos dengan |jump| > 0.5 mdpl: {(jump_df['jump'].abs() > 0.5).sum()}")

print("\n⚠️  Rekomendasi masking lag:")
if (jump_df['jump'].abs() > 1).sum() > 5:
    print("  → Lompatan besar di banyak pos: WAJIB mask lag-1/lag-2/rolling")
    print("    di baris pertama setelah gap (1 Mar 2025 ke atas)")
else:
    print("  → Lompatan relatif kecil: lag mungkin masih acceptable,")
    print("    tapi tetap disarankan mask sebagai precaution")


print("\n" + "=" * 70)
print("EDA TAMBAHAN 6: SIMPLE BASELINE PER POS (LINEAR REGRESSION)")
print("=" * 70)

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

# Fitur yang dipakai di baseline sederhana (semua available, tanpa lag)
baseline_features = [
    'rainfall_max_24h_mm', 'humidity_pct', 'dew_point_c',
    'soil_moisture_7_28cm', 'soil_moisture_0_7cm',
    'cloud_cover_pct', 'temperature_c'
]

# Pastikan kolom bulan & jam ada
merged['hour']  = merged['datetime_parsed'].dt.hour
merged['month'] = merged['datetime_parsed'].dt.month

feature_cols = baseline_features + ['hour', 'month']

print(f"\nFitur baseline: {feature_cols}")
print("Model: Ridge Regression per pos (train 80% awal, test 20% akhir — temporal split)\n")

baseline_results = []
for pos in merged['nama_pos'].unique():
    sub = merged[merged['nama_pos'] == pos].sort_values('datetime_parsed').copy()
    sub = sub.dropna(subset=feature_cols + ['tma_mdpl'])

    if len(sub) < 50:
        print(f"  ⚠️  {pos}: skip (hanya {len(sub)} obs)")
        continue

    # Temporal split: 80% train, 20% test
    split_idx = int(len(sub) * 0.8)
    X_train = sub[feature_cols].iloc[:split_idx]
    y_train = sub['tma_mdpl'].iloc[:split_idx]
    X_test  = sub[feature_cols].iloc[split_idx:]
    y_test  = sub['tma_mdpl'].iloc[split_idx:]

    # Naive baseline: predict mean dari train
    naive_pred  = np.full(len(y_test), y_train.mean())
    naive_rmse  = np.sqrt(mean_squared_error(y_test, naive_pred))

    # Ridge regression
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
    pipe.fit(X_train, y_train)
    ridge_pred = pipe.predict(X_test)
    ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

    improvement = (naive_rmse - ridge_rmse) / naive_rmse * 100

    baseline_results.append({
        'nama_pos':    pos,
        'n_train':     len(X_train),
        'n_test':      len(X_test),
        'naive_rmse':  naive_rmse,
        'ridge_rmse':  ridge_rmse,
        'improvement': improvement
    })

baseline_df = pd.DataFrame(baseline_results).sort_values('ridge_rmse', ascending=False)

print("--- Hasil baseline per pos (sorted by ridge_rmse desc) ---")
print(baseline_df.round(4).to_string(index=False))

print(f"\n--- Summary ---")
print(f"Rata-rata naive RMSE : {baseline_df['naive_rmse'].mean():.4f}")
print(f"Rata-rata ridge RMSE : {baseline_df['ridge_rmse'].mean():.4f}")
print(f"Rata-rata improvement: {baseline_df['improvement'].mean():.1f}%")
print(f"Pos paling susah (ridge RMSE tertinggi): {baseline_df.iloc[0]['nama_pos']} ({baseline_df.iloc[0]['ridge_rmse']:.4f})")
print(f"Pos paling mudah (ridge RMSE terendah): {baseline_df.iloc[-1]['nama_pos']} ({baseline_df.iloc[-1]['ridge_rmse']:.4f})")

print(f"\n--- Pos yang improvement-nya KECIL (<10%) → fitur linear tidak cukup ---")
hard_pos = baseline_df[baseline_df['improvement'] < 10]
print(hard_pos[['nama_pos', 'naive_rmse', 'ridge_rmse', 'improvement']].to_string(index=False))

print(f"\n--- Residual analysis: pos dengan improvement negatif (ridge lebih buruk dari naive) ---")
neg_improvement = baseline_df[baseline_df['improvement'] < 0]
if len(neg_improvement) > 0:
    print(neg_improvement[['nama_pos', 'naive_rmse', 'ridge_rmse', 'improvement']].to_string(index=False))
    print("→ Pos ini kemungkinan spike-dominated, linear model tidak bisa tangkap")
else:
    print("✅ Tidak ada pos yang ridge-nya lebih buruk dari naive")

print("\n" + "=" * 70)
print("SELESAI — EDA TAMBAHAN 1-6")
print("=" * 70)

In [ ]:
pip install geopandas

In [ ]:
import pandas as pd
import numpy as np

# =============================================================
# EDA TAMBAHAN 7-11
# Asumsi: df, merged, env sudah ada dari EDA sebelumnya
# Tambahan: koordinat_pos.csv dibaca di sini
# =============================================================

# ── load koordinat ──────────────────────────────────────────
coords = pd.read_csv("data_pendukung/koordinat_pos.csv")

print("=" * 70)
print("EDA TAMBAHAN 7: ANALISIS KOORDINAT SPASIAL")
print("=" * 70)

print("\n--- Isi koordinat_pos.csv ---")
print(coords.to_string(index=False))

# Merge koordinat ke stats per pos
stats_pos = df.groupby('nama_pos')['tma_mdpl'].agg(
    mean='mean', std='std'
).reset_index()
ac_pos = pd.DataFrame({
    'nama_pos': df['nama_pos'].unique(),
    'ac_lag1': [
        df[df['nama_pos'] == p].sort_values('datetime_parsed')['tma_mdpl'].autocorr(lag=1)
        for p in df['nama_pos'].unique()
    ]
})
lag_opt = (
    lag_df[lag_df['feature'] == 'rainfall_max_24h_mm']
    .loc[lag_df[lag_df['feature'] == 'rainfall_max_24h_mm']
         .groupby('nama_pos')['corr'].idxmax()]
    [['nama_pos', 'lag_6h']]
    .rename(columns={'lag_6h': 'lag_opt_6h'})
)

geo = (coords
       .merge(stats_pos, on='nama_pos')
       .merge(ac_pos, on='nama_pos')
       .merge(lag_opt, on='nama_pos'))

# Urutkan berdasarkan latitude (proxy hulu-hilir: utara vs selatan)
geo_sorted = geo.sort_values('latitude', ascending=False)
print("\n--- Pos diurutkan by latitude (north→south) ---")
print(geo_sorted[['nama_pos', 'latitude', 'longitude',
                   'mean', 'ac_lag1', 'lag_opt_6h']].to_string(index=False))

# Urutkan berdasarkan mean TMA (proxy elevasi)
geo_elev = geo.sort_values('mean', ascending=False)
print("\n--- Pos diurutkan by mean TMA desc (proxy elevasi) ---")
print(geo_elev[['nama_pos', 'latitude', 'longitude',
                 'mean', 'ac_lag1', 'lag_opt_6h']].to_string(index=False))

# Hitung jarak Haversine antar semua pasangan pos
from itertools import combinations

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

dist_rows = []
for (_, r1), (_, r2) in combinations(coords.iterrows(), 2):
    d = haversine(r1['latitude'], r1['longitude'],
                  r2['latitude'], r2['longitude'])
    dist_rows.append({'pos_a': r1['nama_pos'], 'pos_b': r2['nama_pos'], 'dist_km': d})

dist_df = pd.DataFrame(dist_rows)

print("\n--- Top 15 pasangan pos TERDEKAT (km) ---")
print(dist_df.nsmallest(15, 'dist_km').to_string(index=False))

print("\n--- Top 10 pasangan pos TERJAUH (km) ---")
print(dist_df.nlargest(10, 'dist_km').to_string(index=False))

# Cek apakah lag optimal konsisten dengan jarak (hulu-hilir lebih jauh = lag lebih besar)
# Gabung lag optimal dengan jarak untuk top pair dari EDA 2
top_cross_pairs = [
    ('Cepu', 'Sumberrejo'),
    ('Cepu', 'Karanggeneng'),
    ('Bengkelolor', 'Boboh Kali Lamong'),
    ('Jurug', 'Serenan'),
    ('Babat', 'Sumberrejo'),
]
print("\n--- Jarak vs Lag optimal untuk pasangan hulu-hilir EDA 2 ---")
print(f"{'Pos A':<30} {'Pos B':<25} {'Dist (km)':>10} {'Lag opt (jam)':>14}")
print("-" * 82)
for pa, pb in top_cross_pairs:
    row = dist_df[
        ((dist_df['pos_a'] == pa) & (dist_df['pos_b'] == pb)) |
        ((dist_df['pos_a'] == pb) & (dist_df['pos_b'] == pa))
    ]
    d = row['dist_km'].values[0] if len(row) > 0 else np.nan
    # Ambil lag dari cross_lag_df
    lag_row = cross_lag_df[
        ((cross_lag_df['pos_a'] == pa) & (cross_lag_df['pos_b'] == pb)) |
        ((cross_lag_df['pos_a'] == pb) & (cross_lag_df['pos_b'] == pa))
    ]
    if len(lag_row) > 0:
        best_lag = lag_row.loc[lag_row['corr'].idxmax(), 'lag_hours']
    else:
        best_lag = np.nan
    print(f"{pa:<30} {pb:<25} {d:>10.1f} {best_lag:>14.0f}")


print("\n" + "=" * 70)
print("EDA TAMBAHAN 8: HYDRORIVERS — URUTAN HULU-HILIR")
print("=" * 70)

try:
    import geopandas as gpd
    from shapely.geometry import Point

    shp_path = "data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp"
    rivers = gpd.read_file(shp_path)

    print(f"HydroRIVERS shape: {rivers.shape}")
    print(f"Kolom: {list(rivers.columns)}")
    print(f"CRS: {rivers.crs}")
    print(rivers.head(3).to_string())

    # Buat GeoDataFrame dari koordinat pos
    gdf_pos = gpd.GeoDataFrame(
        coords,
        geometry=[Point(lon, lat) for lat, lon in zip(coords['latitude'], coords['longitude'])],
        crs='EPSG:4326'
    )

    # Pastikan CRS sama
    if rivers.crs != gdf_pos.crs:
        rivers = rivers.to_crs(gdf_pos.crs)

    # Snap setiap pos ke sungai terdekat
    # Buffer kecil (0.1 derajat ~11km) lalu intersect
    print("\n--- Mencari sungai terdekat untuk tiap pos ---")
    nearest_rows = []
    for _, pos_row in gdf_pos.iterrows():
        dists = rivers.geometry.distance(pos_row.geometry)
        idx_min = dists.idxmin()
        nearest = rivers.loc[idx_min]
        nearest_rows.append({
            'nama_pos': pos_row['nama_pos'],
            'dist_to_river_deg': dists[idx_min],
            'river_id': nearest.get('HYRIV_ID', np.nan),
            'next_down': nearest.get('NEXT_DOWN', np.nan),
            'ord_flow': nearest.get('ORD_FLOW', np.nan),
            'dis_av_cms': nearest.get('DIS_AV_CMS', np.nan),
        })

    river_pos = pd.DataFrame(nearest_rows).sort_values('dis_av_cms', ascending=False)
    print(river_pos.to_string(index=False))

    print("\n--- Urutan pos by discharge rata-rata (proxy hulu→hilir) ---")
    print(river_pos.sort_values('dis_av_cms')[['nama_pos', 'dis_av_cms', 'ord_flow']].to_string(index=False))

except ImportError:
    print("⚠️  geopandas tidak terinstall.")
    print("    Jalankan: pip install geopandas")
    print("    EDA 8 di-skip, lanjut ke EDA 9.")
except Exception as e:
    print(f"⚠️  Error saat load HydroRIVERS: {e}")
    print("    EDA 8 di-skip, lanjut ke EDA 9.")


print("\n" + "=" * 70)
print("EDA TAMBAHAN 9: RESIDUAL BASELINE PER BULAN & PER JAM")
print("=" * 70)

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

baseline_features = [
    'rainfall_max_24h_mm', 'humidity_pct', 'dew_point_c',
    'soil_moisture_7_28cm', 'soil_moisture_0_7cm',
    'cloud_cover_pct', 'temperature_c'
]
merged['hour']  = merged['datetime_parsed'].dt.hour
merged['month'] = merged['datetime_parsed'].dt.month
feature_cols    = baseline_features + ['hour', 'month']

# Kumpulkan residual dari semua pos (temporal split 80/20)
all_residuals = []
for pos in merged['nama_pos'].unique():
    sub = merged[merged['nama_pos'] == pos].sort_values('datetime_parsed').copy()
    sub = sub.dropna(subset=feature_cols + ['tma_mdpl'])
    if len(sub) < 50:
        continue
    split_idx = int(len(sub) * 0.8)
    X_train = sub[feature_cols].iloc[:split_idx]
    y_train = sub['tma_mdpl'].iloc[:split_idx]
    X_test  = sub[feature_cols].iloc[split_idx:]
    y_test  = sub['tma_mdpl'].iloc[split_idx:]

    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    resid_df = sub.iloc[split_idx:][['datetime_parsed', 'nama_pos', 'tma_mdpl', 'month', 'hour']].copy()
    resid_df['pred']     = preds
    resid_df['residual'] = y_test.values - preds
    resid_df['abs_err']  = np.abs(resid_df['residual'])
    all_residuals.append(resid_df)

resid_all = pd.concat(all_residuals, ignore_index=True)

print("\n--- MAE per bulan (across semua pos) ---")
mae_month = resid_all.groupby('month')['abs_err'].mean().round(4)
print(mae_month.to_frame('mae').T)

print("\n--- MAE per jam observasi (06 / 12 / 18) ---")
mae_hour = resid_all.groupby('hour')['abs_err'].mean().round(4)
print(mae_hour.to_frame('mae').T)

print("\n--- MAE per (bulan × jam) — heatmap table ---")
mae_month_hour = resid_all.groupby(['month', 'hour'])['abs_err'].mean().unstack()
print(mae_month_hour.round(4))

print("\n--- Bulan dengan MAE tertinggi (top 3) ---")
print(mae_month.nlargest(3))
print("\n--- Bulan dengan MAE terendah (top 3) ---")
print(mae_month.nsmallest(3))

print("\n--- Per pos: MAE per bulan (pos paling susah dari EDA 6) ---")
hard_pos = ['Peren', 'Floodway Bridge C', 'Kali Anyar - Kreteg Abang',
            'Kali Pepe - PTPN', 'Wonogiri Dam']
for pos in hard_pos:
    sub_r = resid_all[resid_all['nama_pos'] == pos]
    if len(sub_r) == 0:
        continue
    print(f"\n  {pos}:")
    print(sub_r.groupby('month')['abs_err'].mean().round(4).to_frame('mae').T.to_string())


print("\n" + "=" * 70)
print("EDA TAMBAHAN 10: FLOODWAY BRIDGE C — LEVEL SHIFT SEBELUM vs SESUDAH GAP")
print("=" * 70)

pos_fb = 'Floodway Bridge C'
sub_fb = df[df['nama_pos'] == pos_fb].sort_values('datetime_parsed').copy()

gap_fb_start = pd.Timestamp('2023-06-16')
gap_fb_end   = pd.Timestamp('2024-01-01')

before_gap = sub_fb[sub_fb['datetime_parsed'] < gap_fb_start]
after_gap  = sub_fb[sub_fb['datetime_parsed'] >= gap_fb_end]

print(f"\nPeriode SEBELUM gap ({before_gap['datetime_parsed'].min().date()} → {before_gap['datetime_parsed'].max().date()}):")
print(before_gap['tma_mdpl'].describe().round(4))

print(f"\nPeriode SESUDAH gap ({after_gap['datetime_parsed'].min().date()} → {after_gap['datetime_parsed'].max().date()}):")
print(after_gap['tma_mdpl'].describe().round(4))

# T-test apakah mean beda signifikan
from scipy.stats import ttest_ind, ks_2samp

t_stat, t_pval = ttest_ind(before_gap['tma_mdpl'].dropna(),
                            after_gap['tma_mdpl'].dropna())
ks_stat, ks_pval = ks_2samp(before_gap['tma_mdpl'].dropna(),
                              after_gap['tma_mdpl'].dropna())

print(f"\n--- Uji statistik distribusi sebelum vs sesudah gap ---")
print(f"T-test  : t={t_stat:.4f}, p={t_pval:.6f} {'→ BEDA SIGNIFIKAN' if t_pval < 0.05 else '→ tidak signifikan'}")
print(f"KS-test : D={ks_stat:.4f}, p={ks_pval:.6f} {'→ DISTRIBUSI BERBEDA' if ks_pval < 0.05 else '→ distribusi sama'}")

print(f"\nDelta mean (after - before): {after_gap['tma_mdpl'].mean() - before_gap['tma_mdpl'].mean():.4f} mdpl")
print(f"Delta std  (after - before): {after_gap['tma_mdpl'].std()  - before_gap['tma_mdpl'].std():.4f}")

# Distribusi per bulan sebelum vs sesudah (buat cek seasonal confounding)
print("\n--- Mean TMA per bulan SEBELUM gap ---")
print(before_gap.groupby(before_gap['datetime_parsed'].dt.month)['tma_mdpl'].mean().round(4))
print("\n--- Mean TMA per bulan SESUDAH gap ---")
print(after_gap.groupby(after_gap['datetime_parsed'].dt.month)['tma_mdpl'].mean().round(4))

# Step function check: berapa kali lompatan besar dalam 1 step (6 jam)?
sub_fb['delta_step'] = sub_fb['tma_mdpl'].diff().abs()
sub_fb['time_delta'] = sub_fb['datetime_parsed'].diff().dt.total_seconds() / 3600
valid_steps = sub_fb[sub_fb['time_delta'] == 6]

print(f"\n--- Distribusi |delta| antar step valid (6 jam) ---")
print(valid_steps['delta_step'].describe().round(4))
print(f"\nJumlah step dengan |delta| > 1 mdpl: {(valid_steps['delta_step'] > 1).sum()}")
print(f"Jumlah step dengan |delta| > 2 mdpl: {(valid_steps['delta_step'] > 2).sum()}")
print(f"Jumlah step dengan |delta| > 5 mdpl: {(valid_steps['delta_step'] > 5).sum()}")
print(f"\nTop 10 lompatan terbesar (valid steps only):")
print(valid_steps.nlargest(10, 'delta_step')[
    ['datetime', 'tma_mdpl', 'delta_step']].to_string(index=False))


print("\n" + "=" * 70)
print("EDA TAMBAHAN 11: WONOGIRI DAM — STEP-FUNCTION & POLA OPERASIONAL")
print("=" * 70)

pos_wd = 'Wonogiri Dam'
sub_wd = df[df['nama_pos'] == pos_wd].sort_values('datetime_parsed').copy()
sub_wd['delta_step'] = sub_wd['tma_mdpl'].diff()
sub_wd['abs_delta']  = sub_wd['delta_step'].abs()
sub_wd['time_delta'] = sub_wd['datetime_parsed'].diff().dt.total_seconds() / 3600
valid_wd = sub_wd[sub_wd['time_delta'] == 6].copy()

print(f"\n--- Statistik TMA Wonogiri Dam ---")
print(sub_wd['tma_mdpl'].describe().round(4))

print(f"\n--- Distribusi |delta| antar step valid (6 jam) ---")
print(valid_wd['abs_delta'].describe().round(4))

# Threshold step-function: lompatan besar yang mungkin operasional
for thresh in [0.5, 1.0, 2.0, 3.0]:
    n = (valid_wd['abs_delta'] > thresh).sum()
    pct = n / len(valid_wd) * 100
    print(f"  |delta| > {thresh} mdpl: {n:4d} steps ({pct:.2f}%)")

print(f"\nTop 15 lompatan terbesar (Wonogiri Dam, valid 6-jam steps):")
print(valid_wd.nlargest(15, 'abs_delta')[
    ['datetime', 'tma_mdpl', 'delta_step', 'abs_delta']].to_string(index=False))

# Cek apakah lompatan besar Wonogiri berkorelasi dengan rainfall
# (lompatan besar tapi rainfall rendah = indikasi operasional, bukan hidrologi)
wd_merged = merged[merged['nama_pos'] == pos_wd].sort_values('datetime_parsed').copy()
wd_merged['abs_delta'] = wd_merged['tma_mdpl'].diff().abs()
wd_merged['time_delta'] = wd_merged['datetime_parsed'].diff().dt.total_seconds() / 3600
wd_valid = wd_merged[wd_merged['time_delta'] == 6].copy()

print(f"\n--- Step besar Wonogiri vs rainfall (operasional vs hidrologi?) ---")
thresh_op = 1.0
big_steps = wd_valid[wd_valid['abs_delta'] > thresh_op]
small_steps = wd_valid[wd_valid['abs_delta'] <= thresh_op]

print(f"Step besar (|delta|>{thresh_op}): n={len(big_steps)}")
print(f"  Mean rainfall_max_24h: {big_steps['rainfall_max_24h_mm'].mean():.4f}")
print(f"  Mean soil_moisture_7_28cm: {big_steps['soil_moisture_7_28cm'].mean():.4f}")
print(f"\nStep kecil (|delta|<={thresh_op}): n={len(small_steps)}")
print(f"  Mean rainfall_max_24h: {small_steps['rainfall_max_24h_mm'].mean():.4f}")
print(f"  Mean soil_moisture_7_28cm: {small_steps['soil_moisture_7_28cm'].mean():.4f}")

# Pola musiman lompatan Wonogiri
print(f"\n--- Frekuensi step besar (|delta|>{thresh_op}) per bulan ---")
big_steps_month = big_steps.copy()
big_steps_month['month'] = big_steps_month['datetime_parsed'].dt.month
print(big_steps_month['month'].value_counts().sort_index())

# Tren tahunan Wonogiri
print(f"\n--- Mean TMA Wonogiri per tahun (konfirmasi tren naik) ---")
sub_wd['year'] = sub_wd['datetime_parsed'].dt.year
print(sub_wd.groupby('year')['tma_mdpl'].agg(['mean', 'std', 'min', 'max']).round(4))

print("\n" + "=" * 70)
print("SELESAI — EDA TAMBAHAN 7-11")
print("=" * 70)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp

# =============================================================
# STAGE 3 — STEP 1: TRAIN vs TEST DISTRIBUTION DRIFT
# env sudah ada dari EDA sebelumnya
# =============================================================

# Split env berdasarkan periode train vs test
train_cutoff = pd.Timestamp('2025-09-19')

env['datetime_parsed'] = pd.to_datetime(env['datetime'])
env_train = env[env['datetime_parsed'] <  train_cutoff].copy()
env_test  = env[env['datetime_parsed'] >= train_cutoff].copy()

print(f"env_train: {env_train.shape} | {env_train['datetime_parsed'].min().date()} → {env_train['datetime_parsed'].max().date()}")
print(f"env_test : {env_test.shape}  | {env_test['datetime_parsed'].min().date()} → {env_test['datetime_parsed'].max().date()}")

# Fitur yang akan dicek drift-nya
# Exclude: solar_radiation (sudah confirmed -999 di test), 
#          nino_34 (missing Mei 2026),
#          duplikat, statis
drift_cols = [
    'rainfall_mm',
    'rainfall_max_24h_mm',
    'humidity_pct',
    'wind_direction_deg',
    'dew_point_c',
    'cloud_cover_pct',
    'temperature_c',
    'wind_speed_kmh',
    'soil_moisture_0_7cm',
    'soil_moisture_7_28cm',
    'soil_moisture_28_100cm',
    'soil_moisture_100_255cm',
    'surface_pressure_hpa',
    'pressure_msl_hpa',
    'rmm1',
    'rmm2',
    'mjo_phase',
    'mjo_amplitude',
    'mjo_active',
    'nino_34',
]

print("\n--- KS-test drift: train vs test (per fitur) ---")
print(f"{'Feature':<30} {'KS stat':>8} {'p-value':>12} {'Mean train':>12} {'Mean test':>11} {'Drift?':>8}")
print("-" * 90)

drift_results = []
for col in drift_cols:
    tr = env_train[col].dropna()
    te = env_test[col].dropna()
    if len(tr) == 0 or len(te) == 0:
        print(f"{col:<30} {'N/A':>8} {'N/A':>12} {'N/A':>12} {'N/A':>11} {'NO DATA':>8}")
        continue
    ks_stat, ks_pval = ks_2samp(tr, te)
    flag = '🔴 DRIFT' if ks_pval < 0.05 and ks_stat > 0.1 else ('🟡 MILD' if ks_pval < 0.05 else '✅ OK')
    drift_results.append({
        'feature': col, 'ks_stat': ks_stat, 'ks_pval': ks_pval,
        'mean_train': tr.mean(), 'mean_test': te.mean(),
        'std_train': tr.std(), 'std_test': te.std(), 'flag': flag
    })
    print(f"{col:<30} {ks_stat:>8.4f} {ks_pval:>12.2e} {tr.mean():>12.4f} {te.mean():>11.4f} {flag:>8}")

drift_df = pd.DataFrame(drift_results)

print(f"\n--- Ringkasan ---")
print(f"Total fitur dicek : {len(drift_results)}")
print(f"🔴 DRIFT parah    : {(drift_df['flag'] == '🔴 DRIFT').sum()}")
print(f"🟡 MILD drift     : {(drift_df['flag'] == '🟡 MILD').sum()}")
print(f"✅ OK             : {(drift_df['flag'] == '✅ OK').sum()}")

print(f"\n--- Fitur dengan drift PARAH (KS > 0.1, p < 0.05) ---")
bad = drift_df[drift_df['flag'] == '🔴 DRIFT'].sort_values('ks_stat', ascending=False)
print(bad[['feature', 'ks_stat', 'mean_train', 'mean_test']].to_string(index=False))

# Cek seasonal confounding: apakah drift karena bulan berbeda?
# Train berakhir Sep, test mulai Sep → bulan Oct-Mei punya referensi di train
# tapi proporsinya berbeda (test: Sep-Mei, train: Jan-Sep tiap tahun)
print(f"\n--- Distribusi bulan di train env vs test env ---")
env_train['month'] = env_train['datetime_parsed'].dt.month
env_test['month']  = env_test['datetime_parsed'].dt.month
train_month_dist = env_train['month'].value_counts(normalize=True).sort_index()
test_month_dist  = env_test['month'].value_counts(normalize=True).sort_index()
month_comp = pd.DataFrame({
    'train_pct': (train_month_dist * 100).round(1),
    'test_pct':  (test_month_dist  * 100).round(1)
}).fillna(0)
month_comp['diff'] = (month_comp['test_pct'] - month_comp['train_pct']).round(1)
print(month_comp.to_string())

In [ ]:
# =============================================================
# STAGE 3 — STEP 2: MISSING VALUE PATTERN (MCAR/MAR/MNAR)
# =============================================================

print("=" * 70)
print("STAGE 3 — STEP 2: MISSING VALUE PATTERN")
print("=" * 70)

# Kolom yang punya missing values (dari EDA sebelumnya)
missing_cols = [
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm',
    'soil_moisture_28_100cm', 'soil_moisture_100_255cm',
    'surface_pressure_hpa', 'pressure_msl_hpa',
    'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active',
    'nino_34'
]

print("\n--- Missing count per kolom (env full) ---")
for col in missing_cols:
    n_miss = env[col].isna().sum()
    pct    = n_miss / len(env) * 100
    # Cek apakah missing hanya di train atau test
    n_miss_train = env_train[col].isna().sum()
    n_miss_test  = env_test[col].isna().sum()
    print(f"{col:<30} total={n_miss:6d} ({pct:.2f}%)  "
          f"train={n_miss_train}  test={n_miss_test}")

# --- Diagnosis MCAR/MAR/MNAR ---
print("\n--- Diagnosis mekanisme missing ---")

# soil_moisture & pressure & MJO: missing 720 baris = 30 pos × 24 jam di 2026-05-18
print("\n[1] soil_moisture / pressure / MJO (720 missing):")
miss_rows = env[env['soil_moisture_0_7cm'].isna()]
print(f"  Tanggal unik: {miss_rows['datetime_parsed'].dt.date.unique()}")
print(f"  Pos unik   : {miss_rows['nama_pos'].nunique()}")
print(f"  → Verdict  : MCAR — cutoff artifact, 1 hari terakhir, semua pos")

# nino_34: missing 12.960 baris = seluruh Mei 2026
print("\n[2] nino_34 (12.960 missing):")
miss_nino = env[env['nino_34'].isna()]
print(f"  Periode: {miss_nino['datetime_parsed'].min()} → {miss_nino['datetime_parsed'].max()}")
print(f"  Pos unik: {miss_nino['nama_pos'].nunique()}")
nino_vals = env[env['nino_34'].notna()].groupby(
    env[env['nino_34'].notna()]['datetime_parsed'].dt.to_period('M')
)['nino_34'].first().tail(6)
print(f"  Nilai nino_34 bulan-bulan terakhir sebelum missing:")
print(nino_vals)
print(f"  → Verdict  : MCAR — data belum tersedia (future data), bukan missing karena nilai")

# solar_radiation: sudah confirmed -999 = missing, bukan MCAR biasa
print("\n[3] solar_radiation_mj_m2 (-999 sentinel):")
solar_miss = env[env['solar_radiation_mj_m2'] == -999]
solar_valid = env[env['solar_radiation_mj_m2'] != -999]
print(f"  Period -999 : {solar_miss['datetime_parsed'].min().date()} → {solar_miss['datetime_parsed'].max().date()}")
print(f"  n rows -999 : {len(solar_miss)} ({len(solar_miss)/len(env)*100:.1f}%)")
print(f"  % di train  : {(env_train['solar_radiation_mj_m2'] == -999).mean()*100:.1f}%")
print(f"  % di test   : {(env_test['solar_radiation_mj_m2'] == -999).mean()*100:.1f}%")
print(f"  → Verdict   : MNAR — missing HANYA terjadi di periode future (test period dominan)")
print(f"                Train 100% valid, test ~60% missing → structural mismatch")

# Cek apakah ada missing tersembunyi di fitur lain
print("\n--- Cek missing tersembunyi di kolom lain (nilai 0 yg mencurigakan) ---")
suspect_zero_cols = ['rainfall_mm', 'wind_speed_kmh', 'solar_radiation_mj_m2']
for col in suspect_zero_cols:
    n_zero = (env[col] == 0).sum()
    pct_zero = n_zero / len(env) * 100
    print(f"  {col:<30}: {n_zero:7d} zeros ({pct_zero:.1f}%)")

# Cek train.csv: missing di target?
print("\n--- Missing di train target (tma_mdpl) ---")
print(f"  Missing count: {df['tma_mdpl'].isna().sum()}")
print(f"  Nilai <= 0   : {(df['tma_mdpl'] <= 0).sum()} (sudah tercatat sebagai anomali Flag 9)")

# Summary imputation strategy
print("\n--- Rekomendasi imputation strategy ---")
print("""
  soil_moisture / pressure / MJO (720 rows, 2026-05-18):
  → Forward-fill dari hari sebelumnya (2026-05-17)
  → Low risk: 1 hari, semua fitur bergerak lambat

  nino_34 (Mei 2026, 12.960 rows):
  → Forward-fill dari April 2026
  → Acceptable: ENSO index bergerak lambat (bulanan)
  → Tanda bahwa test period = La Niña berlanjut

  solar_radiation_mj_m2 (-999, ~60% test):
  → JANGAN impute — structural mismatch train-test
  → Rekomendasi: DROP kolom ini
  → Proxy: cloud_cover_pct (tersedia penuh, corr negatif dengan solar)

  tma_mdpl anomali (5 baris <= 0):
  → Impute dengan forward-fill per pos
  → Jumlah sangat kecil, dampak minimal
""")

In [ ]:
# Klarifikasi definisi rainfall_max_24h_mm
# Cek apakah ini rolling backward atau forward dari timestamp
import pandas as pd

env = pd.read_csv("data_pendukung/data_lingkungan.csv")
env['datetime_parsed'] = pd.to_datetime(env['datetime'])

# Ambil 1 pos, 3 hari pertama
sample = env[env['nama_pos'] == 'Arjowinangun - Pacitan'].sort_values('datetime_parsed').head(72)

# Cek: apakah rainfall_max_24h_mm[t] == max(rainfall_mm[t-24h:t]) atau max(rainfall_mm[t:t+24h])?
sample['rolling_max_24h_backward'] = sample['rainfall_mm'].rolling(24).max()
sample['rolling_max_24h_forward'] = sample['rainfall_mm'][::-1].rolling(24).max()[::-1]

print(sample[['datetime', 'rainfall_mm', 'rainfall_max_24h_mm',
              'rolling_max_24h_backward', 'rolling_max_24h_forward']].head(30))

In [ ]:
# Investigasi manual Floodway Bridge C sekitar 15 Jun 2025
floodway = df[df['nama_pos'] == 'Floodway Bridge C'].sort_values('datetime_parsed')

# Lihat window ±7 hari sekitar outlier
mask = (
    (floodway['datetime_parsed'] >= '2025-06-08') &
    (floodway['datetime_parsed'] <= '2025-06-22')
)
print(floodway[mask][['datetime', 'tma_mdpl']])

# Cek rainfall di periode yang sama
env_flood = env_matched[env_matched['nama_pos'] == 'Floodway Bridge C']
env_mask = (
    (env_flood['datetime_parsed'] >= '2025-06-08') &
    (env_flood['datetime_parsed'] <= '2025-06-22')
)
print(env_flood[env_mask][['datetime', 'rainfall_mm', 'rainfall_max_24h_mm']])

In [ ]:
# Remove 1 baris outlier Floodway Bridge C
outlier_mask = (
    (df['nama_pos'] == 'Floodway Bridge C') &
    (df['datetime'] == '2025-06-15 18:00:00')
)

print(f"Baris yang di-remove: {outlier_mask.sum()}")  # harus 1
df = df[~outlier_mask].reset_index(drop=True)

# Verifikasi max Floodway setelah remove
print(df[df['nama_pos'] == 'Floodway Bridge C']['tma_mdpl'].max())
# Ekspektasi: ~4.7 (bukan 47.0)

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
env = pd.read_csv("data_pendukung/data_lingkungan.csv")

# Parse datetime
df['datetime_parsed'] = pd.to_datetime(df['datetime'])
test[['datetime_str', 'nama_pos']] = test['id'].str.split(' - ', n=1, expand=True)
test['datetime_parsed'] = pd.to_datetime(test['datetime_str'])
env['datetime_parsed'] = pd.to_datetime(env['datetime'])

# Remove outlier Floodway (sudah dilakukan, tapi safeguard)
outlier_mask = (
    (df['nama_pos'] == 'Floodway Bridge C') &
    (df['datetime'] == '2025-06-15 18:00:00')
)
df = df[~outlier_mask].reset_index(drop=True)

print("Train shape:", df.shape)
print("Test shape:", test.shape)
print("Env shape:", env.shape)

In [ ]:
# Sort env
env = env.sort_values(['nama_pos', 'datetime_parsed']).reset_index(drop=True)

# Forward-fill nino_34 (Mei 2026 missing)
env['nino_34'] = env.groupby('nama_pos')['nino_34'].ffill()

# Drop kolom yang sudah diputuskan
DROP_COLS = [
    'rainfall_openmeteo_mm',
    'solar_radiation_mj_m2',
    'built_surface_m2',
    'landcover_class',
    'landcover_name',
    'mjo_active',
]
env = env.drop(columns=[c for c in DROP_COLS if c in env.columns])

# Rolling rainfall 48h dan 72h (dari rainfall_mm per jam)
env['rolling_rain_48h'] = (
    env.groupby('nama_pos')['rainfall_mm']
    .transform(lambda x: x.rolling(48, min_periods=1).sum())
)
env['rolling_rain_72h'] = (
    env.groupby('nama_pos')['rainfall_mm']
    .transform(lambda x: x.rolling(72, min_periods=1).sum())
)

# Sample env hanya di jam 06/12/18 untuk match target
env_matched = env[
    env['datetime_parsed'].dt.hour.isin([6, 12, 18])
].copy()

print("Env matched shape:", env_matched.shape)
print("nino_34 missing setelah ffill:", env_matched['nino_34'].isnull().sum())
print("rolling_rain_72h sample:\n", env_matched[['datetime', 'nama_pos', 'rolling_rain_48h', 'rolling_rain_72h']].head())

In [ ]:
# Kolom env yang akan dipakai
ENV_FEATURES = [
    'datetime', 'nama_pos',
    'rainfall_mm', 'humidity_pct', 'wind_direction_deg',
    'dew_point_c', 'cloud_cover_pct', 'temperature_c',
    'wind_speed_kmh', 'rainfall_max_24h_mm',
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm',
    'soil_moisture_28_100cm', 'soil_moisture_100_255cm',
    'surface_pressure_hpa', 'pressure_msl_hpa',
    'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'nino_34',
    'rolling_rain_48h', 'rolling_rain_72h',
]

env_slim = env_matched[[c for c in ENV_FEATURES if c in env_matched.columns]]

# Merge train
train = df.merge(env_slim, on=['datetime', 'nama_pos'], how='left')

# Merge test
test = test.merge(env_slim, on=['datetime', 'nama_pos'],
                  left_on=['datetime_str', 'nama_pos'],
                  right_on=['datetime', 'nama_pos'],
                  how='left')

print("Train after merge:", train.shape)
print("Test after merge:", test.shape)
print("Train missing setelah merge:\n", train.isnull().sum()[train.isnull().sum() > 0])
print("Test missing setelah merge:\n", test.isnull().sum()[test.isnull().sum() > 0])